# Tensor Backward Derivations

`tensorgrad` is built on one principle: every gradient rule in the engine is
derived, not looked up. Before writing a single line of `engine.py` or
`functional.py`, I worked out the backward pass for each operation by hand
starting from the scalar chain rule and working up to matrix expressions.

This notebook records those derivations. For each operation: the math first,
then a NumPy implementation of the gradient rule, then a central-difference
finite-difference check to confirm it. No PyTorch, no autograd library just
the definition of a derivative applied to matrices.


In [19]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
sys.path.insert(0, '..')

import numpy as np
from tensorgrad.engine import Tensor, _unbroadcast
from tensorgrad.functional import softmax, cross_entropy, layer_norm, embedding

# ---------------------------------------------------------------------------
# Central-difference numerical gradient
# ---------------------------------------------------------------------------

def numerical_grad(f, x, eps=1e-5):
    """
    Compute df/dx numerically via central differences

    f : callable (np.ndarray) => scalar
    x : np.ndarray of any shape
    Returns an array of the same shape as x
    """
    grad = np.zeros_like(x, dtype=np.float64)
    it   = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx      = it.multi_index
        old      = float(x[idx])
        x[idx]   = old + eps
        fp       = float(f(x))
        x[idx]   = old - eps
        fm       = float(f(x))
        x[idx]   = old           # restore
        grad[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad

# ---------------------------------------------------------------------------
# grad_check: compare autograd gradient to finite-difference estimate
# ---------------------------------------------------------------------------

def grad_check(name, tensor, numerical):
    """
    Compare tensor.grad (autograd) to a numerical estimate
    name      : label for the printout
    tensor    : Tensor whose .grad we inspect
    numerical : np.ndarray of the same shape central-difference estimate
    """
    max_err = np.abs(tensor.grad - numerical).max()
    ok      = np.allclose(tensor.grad, numerical, atol=1e-4, rtol=1e-4)
    print(f'{name:40s} | match: {str(ok):5s} | maxdiff: {max_err:.2e}')


---
## 1. Matrix Multiply

Given $C = A W$ where $A \in \mathbb{R}^{m \times k}$ and $W \in \mathbb{R}^{k \times n}$,
and an upstream gradient $\bar{C} \in \mathbb{R}^{m \times n}$, we need $\bar{A}$ and $\bar{W}$.

A single entry $A_{ij}$ touches $L$ through every element in row $i$ of $C$:

$$C_{il} = \sum_j A_{ij} W_{jl} \implies \frac{\partial L}{\partial A_{ij}} = \sum_l \bar{C}_{il} W_{jl}$$

That sum is the $(i,j)$ entry of $\bar{C} W^\top$, so $\bar{A} = \bar{C} W^\top$.
The same argument on $W_{jl}$ (it appears in every row of $C$, not every column) gives:

$$\frac{\partial L}{\partial W_{jl}} = \sum_i A_{ij} \bar{C}_{il} \implies \bar{W} = A^\top \bar{C}$$

When $A$ has shape $(B, T, k)$ and $W$ is a shared weight $(k, n)$, the same formulas
hold slice-by-slice, but $\bar{W}$ accumulates across the batch `_unbroadcast` handles that sum.

In [20]:
# ------------------------------------------------------------------
# 2-D case: C = A @ B  =>  dA = dC @ B.T,  dB = A.T @ dC
# ------------------------------------------------------------------
rng = np.random.default_rng(0)

A_np = rng.standard_normal((3, 4))
B_np = rng.standard_normal((4, 5))

A = Tensor(A_np.copy())
B = Tensor(B_np.copy())
C = A @ B                           # (3, 5)
C.backward()                        # seeds dL/dC = ones(3,5)

# Numerical checks: fix B and vary A, then fix A and vary B.
f_A = lambda a: (a @ B_np).sum()
f_B = lambda b: (A_np @ b).sum()

num_A = numerical_grad(f_A, A_np.copy())
num_B = numerical_grad(f_B, B_np.copy())

grad_check('matmul dA (2D)', A, num_A)
grad_check('matmul dB (2D)', B, num_B)


matmul dA (2D)                           | match: True  | maxdiff: 4.44e-11
matmul dB (2D)                           | match: True  | maxdiff: 3.35e-11


In [21]:
# ------------------------------------------------------------------
# Batched case: A (B,T,C) @ W (C,H)  =>  dW summed over batch dim
# ------------------------------------------------------------------
B_dim, T, C_dim, H = 2, 3, 4, 5

A_np3 = rng.standard_normal((B_dim, T, C_dim))
W_np  = rng.standard_normal((C_dim, H))

A3 = Tensor(A_np3.copy())
W  = Tensor(W_np.copy())
C3 = A3 @ W                         # (B, T, H)
C3.backward()

f_A3 = lambda a: (a @ W_np).sum()
f_W  = lambda w: (A_np3 @ w).sum()

num_A3 = numerical_grad(f_A3, A_np3.copy())
num_W  = numerical_grad(f_W,  W_np.copy())

grad_check('matmul dA (batched 3D)', A3, num_A3)
grad_check('matmul dW (shared weight, batched)', W, num_W)


matmul dA (batched 3D)                   | match: True  | maxdiff: 1.31e-10
matmul dW (shared weight, batched)       | match: True  | maxdiff: 2.10e-10


---
## 2. Broadcasting and the Unbroadcast Rule

When NumPy broadcasts a tensor, it replicates values along axes that don't
align silently, in the forward pass. The backward pass has to undo that:
if a value was reused $n$ times, its gradient is the sum of all $n$ upstream
contributions.

Two situations come up:

- `grad` has more axes than the original tensor (extra leading dimensions were
  broadcast in). Sum over those leading axes until the ranks match.
- The original had a size-1 dimension that was stretched to size $n$.
  Sum over that axis with `keepdims=True` to get back to shape `(1, ...)`.

`_unbroadcast(grad, shape)` handles both cases given the original shape.

The bias in a linear layer is the clearest example: $b \in \mathbb{R}^{C}$ is
added to every row of an $(N, C)$ output, so $\bar{b} = \sum_i \bar{C}_{i,:}$
a sum over the batch axis. The three code cells below check all three
broadcast shapes that appear in the engine.

In [22]:
# ------------------------------------------------------------------
# Case 1: C = A + b   where b is a bias vector (1-D broadcast)
# ------------------------------------------------------------------
A_np = rng.standard_normal((4, 3))
b_np = rng.standard_normal((3,))

A = Tensor(A_np.copy())
b = Tensor(b_np.copy())
C = A + b                           # b broadcast over rows
C.backward()

f_A = lambda a: (a + b_np).sum()
f_b = lambda bb: (A_np + bb).sum()

grad_check('add: dA (bias broadcast)', A, numerical_grad(f_A, A_np.copy()))
grad_check('add: db (bias vector)',    b, numerical_grad(f_b, b_np.copy()))


add: dA (bias broadcast)                 | match: True  | maxdiff: 3.79e-11
add: db (bias vector)                    | match: True  | maxdiff: 2.62e-11


In [23]:
# ------------------------------------------------------------------
# Case 2: C = A * s   where s is a scalar Tensor
# ------------------------------------------------------------------
s_np = np.array(2.5)
A_np = rng.standard_normal((3, 4))

A = Tensor(A_np.copy())
s = Tensor(s_np.copy())
C = A * s
C.backward()

f_s = lambda x: (A_np * x).sum()

grad_check('mul: ds (scalar broadcast)', s, numerical_grad(f_s, s_np.copy()))


mul: ds (scalar broadcast)               | match: True  | maxdiff: 7.86e-13


In [24]:
# ------------------------------------------------------------------
# Case 3: A (B,T,C) + bias (C,)  - two levels of broadcast
# ------------------------------------------------------------------
B_dim, T, C_dim = 2, 3, 4
A_np   = rng.standard_normal((B_dim, T, C_dim))
b_np   = rng.standard_normal((C_dim,))

A = Tensor(A_np.copy())
b = Tensor(b_np.copy())
C = A + b
C.backward()

f_b3 = lambda bb: (A_np + bb).sum()
grad_check('add: db (2-level broadcast, C)', b, numerical_grad(f_b3, b_np.copy()))


add: db (2-level broadcast, C)           | match: True  | maxdiff: 1.28e-10


---
## 3. Softmax Backward

$$p_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

In practice we subtract $\max_j x_j$ before exponentiating the output is
identical but the computation stays numerically stable.

To backprop through softmax we need the Jacobian $J_{ij} = \partial p_i / \partial x_j$.
Differentiating the quotient gives:

$$J_{ij} = p_i (\delta_{ij} - p_j)$$

where $\delta_{ij}$ is the Kronecker delta. Given upstream gradient $\bar{p}$,
we want $\bar{x}_j = \sum_i J_{ij} \bar{p}_i$:

$$\bar{x}_j = \sum_i p_i (\delta_{ij} - p_j) \bar{p}_i = p_j \bar{p}_j - p_j \sum_i p_i \bar{p}_i$$

Factor out $p_j$:

$$\boxed{\bar{x} = p \odot \left( \bar{p} - \langle \bar{p},\, p \rangle \right)}$$

No need to materialise the full $n \times n$ Jacobian this is a single
element-wise expression, $O(n)$ in memory and time.

In [25]:
# ------------------------------------------------------------------
# Softmax backward: dx = p * (dout - (dout * p).sum(dim, keepdims=True))
# ------------------------------------------------------------------
x_np = rng.standard_normal((5, 4))    # (N, C)

x   = Tensor(x_np.copy())
out = softmax(x, dim=-1)              # (N, C)
out.backward()                        # upstream = ones

# Numerical: apply softmax and sum all outputs
def f_softmax(xv):
    shifted = xv - xv.max(axis=-1, keepdims=True)
    e = np.exp(shifted)
    p = e / e.sum(axis=-1, keepdims=True)
    return p.sum()

grad_check('softmax dx', x, numerical_grad(f_softmax, x_np.copy()))


softmax dx                               | match: True  | maxdiff: 4.44e-11


---
## 4. Cross-Entropy: Fused Backward

The intuition behind softmax and cross-entropy is covered in
[`language-models-from-scratch`](https://github.com/Nur2424/language-models-from-scratch).
Here I care about the gradient.

The naive approach build cross-entropy step by step through softmax, log,
gather, negate, mean creates a chain of intermediate nodes and, worse,
the backward through `log(p)` divides by $p$. For small probabilities that
is numerically explosive.

Instead, differentiate the whole composition at once. Start by writing the
loss in terms of logits directly:

$$l_i = -\log p_{i,y_i} = -x_{i,y_i} + \log \sum_j e^{x_{ij}}$$

Differentiating with respect to logit $x_{i,c}$:

$$\frac{\partial l_i}{\partial x_{i,c}} = -\mathbf{1}[c = y_i] + p_{i,c}$$

Average over $N$ examples:

$$\boxed{\bar{x}_{i,c} = \frac{p_{i,c} - \mathbf{1}[c = y_i]}{N}}$$

Softmax probabilities minus a one-hot label, divided by $N$. No log, no
division by $p$.

In [26]:
# ------------------------------------------------------------------
# Cross-entropy fused backward: dlogits = (p - one_hot(y)) / N
# ------------------------------------------------------------------
N, C = 8, 16
logits_np = rng.standard_normal((N, C))
targets   = rng.integers(0, C, size=(N,))

logits = Tensor(logits_np.copy())
loss   = cross_entropy(logits, targets)
loss.backward()

def f_ce(lv):
    """Compute cross-entropy loss from raw logits (numpy only)"""
    shifted = lv - lv.max(axis=1, keepdims=True)
    e       = np.exp(shifted)
    p       = e / e.sum(axis=1, keepdims=True)
    return -np.log(p[np.arange(N), targets]).mean()

num_logits = numerical_grad(f_ce, logits_np.copy())
grad_check('cross_entropy dlogits', logits, num_logits)


cross_entropy dlogits                    | match: True  | maxdiff: 2.63e-11


In [27]:
# 3-D case: (B, T, C) logits as in GPT
B_dim, T, C = 2, 6, 16
logits_np3 = rng.standard_normal((B_dim, T, C))
targets3   = rng.integers(0, C, size=(B_dim, T))

logits3 = Tensor(logits_np3.copy())
loss3   = cross_entropy(logits3, targets3)
loss3.backward()

def f_ce3(lv):
    lv2     = lv.reshape(-1, C)
    tgt     = targets3.reshape(-1)
    shifted = lv2 - lv2.max(axis=1, keepdims=True)
    e       = np.exp(shifted)
    p       = e / e.sum(axis=1, keepdims=True)
    return -np.log(p[np.arange(lv2.shape[0]), tgt]).mean()

num_logits3 = numerical_grad(f_ce3, logits_np3.copy())
grad_check('cross_entropy dlogits (3D)', logits3, num_logits3)


cross_entropy dlogits (3D)               | match: True  | maxdiff: 6.33e-11


---
## 5. Layer Norm Backward

$$\hat{x} = \frac{x - \mu}{\sigma}, \quad \text{out} = \gamma \hat{x} + \beta$$

where $\mu = \frac{1}{D}\sum_d x_d$ and $\sigma = \sqrt{\frac{1}{D}\sum_d(x_d - \mu)^2 + \varepsilon}$.
Normalization runs over the last axis (features), not the batch axis.

$x$ enters the computation at three places: directly in $x - \mu$, through
$\mu$ (which depends on $x$), and through $\sigma^2$ (which depends on
$(x-\mu)^2$). Tracing all three paths and collecting terms, the gradient
collapses to a single expression. Let $r = 1/\sigma$ and
$\overline{\hat{x}} = \bar{\text{out}} \odot \gamma$ (gradient through the affine step):

$$\boxed{\bar{x} = \frac{r}{D} \left( D\,\overline{\hat{x}} - \sum_d \overline{\hat{x}}_d - \hat{x} \sum_d \overline{\hat{x}}_d \hat{x}_d \right)}$$

No intermediate tensors for $\mu$ or $\sigma$ one expression covers all three paths.

In [28]:
# ------------------------------------------------------------------
# Layer norm backward: dx, dweight, dbias
# ------------------------------------------------------------------
N, D = 4, 8
x_np  = rng.standard_normal((N, D))
w_np  = rng.standard_normal((D,))
b_np  = rng.standard_normal((D,))

x      = Tensor(x_np.copy())
weight = Tensor(w_np.copy())
bias   = Tensor(b_np.copy())

out = layer_norm(x, weight, bias)
out.backward()

eps = 1e-5

def _ln(xv, wv, bv):
    mu   = xv.mean(axis=-1, keepdims=True)
    var  = ((xv - mu) ** 2).mean(axis=-1, keepdims=True)
    xhat = (xv - mu) / np.sqrt(var + eps)
    return (wv * xhat + bv).sum()

f_x = lambda xv: _ln(xv, w_np, b_np)
f_w = lambda wv: _ln(x_np, wv, b_np)
f_b = lambda bv: _ln(x_np, w_np, bv)

grad_check('layer_norm dx',      x,      numerical_grad(f_x, x_np.copy()))
grad_check('layer_norm dweight', weight, numerical_grad(f_w, w_np.copy()))
grad_check('layer_norm dbias',   bias,   numerical_grad(f_b, b_np.copy()))


layer_norm dx                            | match: True  | maxdiff: 1.58e-10
layer_norm dweight                       | match: True  | maxdiff: 5.65e-11
layer_norm dbias                         | match: True  | maxdiff: 1.07e-10


In [29]:
# 3-D input: (B, T, D) as used in transformer blocks
B_dim, T, D = 2, 5, 8
x_np3  = rng.standard_normal((B_dim, T, D))
w_np3  = rng.standard_normal((D,))
b_np3  = rng.standard_normal((D,))

x3      = Tensor(x_np3.copy())
weight3 = Tensor(w_np3.copy())
bias3   = Tensor(b_np3.copy())
out3    = layer_norm(x3, weight3, bias3)
out3.backward()

f_x3 = lambda xv: _ln(xv, w_np3, b_np3)
f_w3 = lambda wv: _ln(x_np3, wv, b_np3)
f_b3 = lambda bv: _ln(x_np3, w_np3, bv)

grad_check('layer_norm dx (3D)',      x3,      numerical_grad(f_x3, x_np3.copy()))
grad_check('layer_norm dweight (3D)', weight3, numerical_grad(f_w3, w_np3.copy()))
grad_check('layer_norm dbias (3D)',   bias3,   numerical_grad(f_b3, b_np3.copy()))


layer_norm dx (3D)                       | match: True  | maxdiff: 1.07e-09
layer_norm dweight (3D)                  | match: True  | maxdiff: 4.50e-10
layer_norm dbias (3D)                    | match: True  | maxdiff: 3.32e-10


---
## 6. Embedding Backward

An embedding layer is a lookup table. Given indices $\text{idx} \in \mathbb{Z}^T$
and weight matrix $W \in \mathbb{R}^{V \times D}$, the forward pass gathers rows:

$$\text{out}_t = W_{\text{idx}_t,\,:} \quad \Longrightarrow \quad
\frac{\partial L}{\partial W_{v,d}} = \sum_{t\,:\,\text{idx}_t = v} \bar{\text{out}}_{t,d}$$

For each row $v$, accumulate gradients from every position that looked up that row.

The tricky case is repeated indices. If token $v$ appears twice, two upstream
gradients must add into $\bar{W}_{v,:}$. Simple slice assignment
`W.grad[idx] = out.grad` overwrites — the second write kills the first.
`np.add.at(W.grad, idx, out.grad)` accumulates correctly.

In [30]:
# ------------------------------------------------------------------
# Embedding backward with repeated indices
# ------------------------------------------------------------------
V, D = 10, 4
idx_np = np.array([2, 5, 2, 7, 2])   # token 2 appears three times
W_np   = rng.standard_normal((V, D))

W   = Tensor(W_np.copy())
out = embedding(W, idx_np)            # (5, D)
out.backward()

# Numerical: perturb each entry of W and measure loss = out.sum()
def f_W(wv):
    return wv[idx_np].sum()

num_W = numerical_grad(f_W, W_np.copy())
grad_check('embedding dW (repeated indices)', W, num_W)


embedding dW (repeated indices)          | match: True  | maxdiff: 6.41e-11


In [31]:
# Verify the scatter-add counts manually for token 2
# Token 2 appears at positions 0, 2, 4.  Its gradient should be 3.0 * 1.0 = 3.0
# (because all upstream grads are 1.0 from backward()).
print('W.grad[2, :]  =', W.grad[2, :])
print('Expected      = 3.0 (three upstream 1s accumulated)')


W.grad[2, :]  = [3. 3. 3. 3.]
Expected      = 3.0 (three upstream 1s accumulated)


---
## 7. Causal Mask Backward

In a decoder-only transformer, position $t$ must not attend to future positions
$t' > t$. Before the softmax, those attention logits are replaced with $-\infty$:

$$A_{ij} = \begin{cases} x_{ij} & j \leq i \\ -\infty & j > i \end{cases}$$

After softmax, $-\infty$ entries become exactly zero future information never flows.

The backward is simple: $-\infty$ is a constant, it does not depend on $x$, so
the gradient at every masked position is zero. In `masked_fill` the backward
just zeroes `out.grad` at masked positions before accumulating into `self.grad`.

One shape detail: the causal mask is $(T, T)$ but the attention tensor is
$(B,\, n_\text{head},\, T, T)$. `masked_fill` uses `np.broadcast_to` to expand
the mask to the full shape before indexing, so the caller never has to add
dimensions manually.

In [ ]:
# ------------------------------------------------------------------
# masked_fill backward: gradient zero at masked positions
# ------------------------------------------------------------------
B_dim, n_head, T = 2, 2, 5
attn_np = rng.standard_normal((B_dim, n_head, T, T))

# Upper-triangular causal mask (future positions)
mask = np.triu(np.ones((T, T), dtype=bool), k=1)   # (T, T)

# Replace -inf with something finite before backward (softmax would do this;
# here we test masked_fill alone, so fill with 0 to get a finite upstream grad).
out_data_finite = out.data.copy()
out_data_finite[np.broadcast_to(mask, out.data.shape)] = 0.0
out.grad = np.ones_like(out.data)   # upstream = ones
out._backward()

# At masked positions the grad should be 0; elsewhere it should be 1.
mask_4d = np.broadcast_to(mask, (B_dim, n_head, T, T))

print('Max grad at masked positions (expect 0):', np.abs(attn.grad[mask_4d]).max())
print('Max grad at unmasked positions (expect 1):', np.abs(attn.grad[~mask_4d] - 1.0).max())


Max grad at masked positions (expect 0): 0.0
Max grad at unmasked positions (expect 1): 0.0


In [33]:
# ------------------------------------------------------------------
# Numerical confirmation via a finite-loss function
# ------------------------------------------------------------------
B_dim, n_head, T = 1, 1, 4
attn_np2 = rng.standard_normal((B_dim, n_head, T, T))
mask2    = np.triu(np.ones((T, T), dtype=bool), k=1)

attn2 = Tensor(attn_np2.copy())
out2  = attn2.masked_fill(mask2, 0.0)  # fill with 0 (finite) for diff check
out2.backward()

def f_mask(av):
    m4d = np.broadcast_to(mask2, av.shape)
    res = av.copy()
    res[m4d] = 0.0
    return res.sum()

num_attn = numerical_grad(f_mask, attn_np2.copy())
grad_check('masked_fill dx', attn2, num_attn)


masked_fill dx                           | match: True  | maxdiff: 3.79e-11


---
## 8. Full Verification Suite

One `grad_check` call per primitive op in `engine.py` and per function in `functional.py`.
I run every op that has a non-trivial backward closure.  The goal is a clean pass
with `maxdiff` well below `1e-4` everywhere.

The setup is deliberate: small tensors (to keep finite-diff cheap), random seeds
fixed for reproducibility, and a simple scalar loss (`.sum()` or a cross-entropy scalar).


In [34]:
rng = np.random.default_rng(42)
print('=== engine.py primitives ===')
print()

# ---------------------------------------------------------------------------
# __add__
# ---------------------------------------------------------------------------
A_np = rng.standard_normal((3, 4))
b_np = rng.standard_normal((4,))      # broadcast case
A = Tensor(A_np.copy()); b = Tensor(b_np.copy())
(A + b).backward()
grad_check('add: dA', A, numerical_grad(lambda x: (x + b_np).sum(), A_np.copy()))
grad_check('add: db', b, numerical_grad(lambda x: (A_np + x).sum(), b_np.copy()))

# ---------------------------------------------------------------------------
# __mul__
# ---------------------------------------------------------------------------
A_np = rng.standard_normal((3, 4)); B_np = rng.standard_normal((3, 4))
A = Tensor(A_np.copy()); B = Tensor(B_np.copy())
(A * B).backward()
grad_check('mul: dA', A, numerical_grad(lambda x: (x * B_np).sum(), A_np.copy()))
grad_check('mul: dB', B, numerical_grad(lambda x: (A_np * x).sum(), B_np.copy()))

# ---------------------------------------------------------------------------
# __pow__
# ---------------------------------------------------------------------------
x_np = rng.standard_normal((3, 4)) + 0.5   # keep away from 0 for fractional powers
x = Tensor(x_np.copy()); (x ** 3).backward()
grad_check('pow (n=3): dx', x, numerical_grad(lambda v: (v ** 3).sum(), x_np.copy()))
x = Tensor(x_np.copy()); (x ** -1).backward()
grad_check('pow (n=-1): dx', x, numerical_grad(lambda v: (v ** -1).sum(), x_np.copy()))

# ---------------------------------------------------------------------------
# __matmul__
# ---------------------------------------------------------------------------
A_np = rng.standard_normal((4, 5)); B_np = rng.standard_normal((5, 3))
A = Tensor(A_np.copy()); B = Tensor(B_np.copy())
(A @ B).backward()
grad_check('matmul: dA', A, numerical_grad(lambda x: (x @ B_np).sum(), A_np.copy()))
grad_check('matmul: dB', B, numerical_grad(lambda x: (A_np @ x).sum(), B_np.copy()))


=== engine.py primitives ===

add: dA                                  | match: True  | maxdiff: 6.55e-12
add: db                                  | match: True  | maxdiff: 1.97e-11
mul: dA                                  | match: True  | maxdiff: 2.25e-11
mul: dB                                  | match: True  | maxdiff: 2.53e-11
pow (n=3): dx                            | match: True  | maxdiff: 1.98e-10
pow (n=-1): dx                           | match: True  | maxdiff: 1.33e-07
matmul: dA                               | match: True  | maxdiff: 7.49e-11
matmul: dB                               | match: True  | maxdiff: 6.56e-11


In [35]:
print('=== reductions ===')
print()

# ---------------------------------------------------------------------------
# sum
# ---------------------------------------------------------------------------
x_np = rng.standard_normal((3, 4, 2))
x = Tensor(x_np.copy()); x.sum().backward()
grad_check('sum (all): dx', x, numerical_grad(lambda v: v.sum(), x_np.copy()))

x = Tensor(x_np.copy()); x.sum(dim=1).backward()
grad_check('sum (dim=1): dx', x, numerical_grad(lambda v: v.sum(axis=1).sum(), x_np.copy()))

# ---------------------------------------------------------------------------
# mean
# ---------------------------------------------------------------------------
x = Tensor(x_np.copy()); x.mean().backward()
grad_check('mean (all): dx', x, numerical_grad(lambda v: v.mean(), x_np.copy()))

x = Tensor(x_np.copy()); x.mean(dim=-1).backward()
grad_check('mean (dim=-1): dx', x, numerical_grad(lambda v: v.mean(axis=-1).sum(), x_np.copy()))

# ---------------------------------------------------------------------------
# max
# ---------------------------------------------------------------------------
x_np2 = rng.standard_normal((3, 4))    # avoid ties
x = Tensor(x_np2.copy()); x.max(dim=1).backward()
grad_check('max (dim=1): dx', x, numerical_grad(lambda v: v.max(axis=1).sum(), x_np2.copy()))


=== reductions ===

sum (all): dx                            | match: True  | maxdiff: 5.10e-11
sum (dim=1): dx                          | match: True  | maxdiff: 5.10e-11
mean (all): dx                           | match: True  | maxdiff: 2.73e-13
mean (dim=-1): dx                        | match: True  | maxdiff: 2.55e-11
max (dim=1): dx                          | match: True  | maxdiff: 6.55e-12


In [36]:
print('=== element-wise nonlinearities ===')
print()

x_np = rng.standard_normal((3, 4))

# exp
x = Tensor(x_np.copy()); x.exp().backward()
grad_check('exp: dx', x, numerical_grad(lambda v: np.exp(v).sum(), x_np.copy()))

# log (need positive inputs)
xp_np = np.abs(x_np) + 0.1
x = Tensor(xp_np.copy()); x.log().backward()
grad_check('log: dx', x, numerical_grad(lambda v: np.log(v).sum(), xp_np.copy()))

# relu
x = Tensor(x_np.copy()); x.relu().backward()
grad_check('relu: dx', x, numerical_grad(lambda v: np.maximum(0, v).sum(), x_np.copy()))

# tanh
x = Tensor(x_np.copy()); x.tanh().backward()
grad_check('tanh: dx', x, numerical_grad(lambda v: np.tanh(v).sum(), x_np.copy()))


=== element-wise nonlinearities ===

exp: dx                                  | match: True  | maxdiff: 1.60e-10
log: dx                                  | match: True  | maxdiff: 1.01e-09
relu: dx                                 | match: True  | maxdiff: 5.10e-11
tanh: dx                                 | match: True  | maxdiff: 4.40e-11


In [37]:
print('=== shape ops ===')
print()

x_np = rng.standard_normal((2, 3, 4))

# reshape
x = Tensor(x_np.copy()); x.reshape(6, 4).backward()
grad_check('reshape: dx', x, numerical_grad(lambda v: v.reshape(6, 4).sum(), x_np.copy()))

# transpose (last two axes)
x = Tensor(x_np.copy()); x.T.backward()
grad_check('transpose (-2,-1): dx', x, numerical_grad(lambda v: v.swapaxes(-2,-1).sum(), x_np.copy()))

# transpose (arbitrary axes)
x = Tensor(x_np.copy()); x.transpose(0, 2).backward()
grad_check('transpose (0,2): dx', x, numerical_grad(lambda v: v.swapaxes(0,2).sum(), x_np.copy()))


=== shape ops ===

reshape: dx                              | match: True  | maxdiff: 5.10e-11
transpose (-2,-1): dx                    | match: True  | maxdiff: 5.10e-11
transpose (0,2): dx                      | match: True  | maxdiff: 5.10e-11


In [38]:
print('=== indexing / cat ===')
print()

W_np  = rng.standard_normal((10, 4))
idx   = np.array([1, 3, 1, 7])           # repeated index 1
W = Tensor(W_np.copy()); W[idx].backward()
grad_check('getitem: dW (repeated)', W, numerical_grad(lambda w: w[idx].sum(), W_np.copy()))

# cat
A_np = rng.standard_normal((3, 4)); B_np = rng.standard_normal((5, 4))
A = Tensor(A_np.copy()); B = Tensor(B_np.copy())
Tensor.cat([A, B], dim=0).backward()
grad_check('cat dim=0: dA', A, numerical_grad(lambda a: np.concatenate([a, B_np], axis=0).sum(), A_np.copy()))
grad_check('cat dim=0: dB', B, numerical_grad(lambda b: np.concatenate([A_np, b], axis=0).sum(), B_np.copy()))


=== indexing / cat ===

getitem: dW (repeated)                   | match: True  | maxdiff: 1.57e-11
cat dim=0: dA                            | match: True  | maxdiff: 3.79e-11
cat dim=0: dB                            | match: True  | maxdiff: 3.79e-11


In [39]:
print('=== functional.py ===')
print()

# softmax
x_np = rng.standard_normal((4, 8))
x = Tensor(x_np.copy())
softmax(x, dim=-1).backward()
def f_sm(v):
    s = v - v.max(axis=-1, keepdims=True)
    e = np.exp(s); return (e / e.sum(axis=-1, keepdims=True)).sum()
grad_check('softmax: dx', x, numerical_grad(f_sm, x_np.copy()))

# cross_entropy
N_ce, C_ce = 6, 10
lg_np = rng.standard_normal((N_ce, C_ce))
tgt   = rng.integers(0, C_ce, size=(N_ce,))
lg = Tensor(lg_np.copy()); cross_entropy(lg, tgt).backward()
def f_ce_suite(v):
    s = v - v.max(axis=1, keepdims=True); e = np.exp(s); p = e / e.sum(axis=1, keepdims=True)
    return -np.log(p[np.arange(N_ce), tgt]).mean()
grad_check('cross_entropy: dlogits', lg, numerical_grad(f_ce_suite, lg_np.copy()))

# layer_norm
N_ln, D_ln = 4, 6
x_np = rng.standard_normal((N_ln, D_ln))
w_np = rng.standard_normal((D_ln,)); b_np2 = rng.standard_normal((D_ln,))
x = Tensor(x_np.copy()); w = Tensor(w_np.copy()); b2 = Tensor(b_np2.copy())
layer_norm(x, w, b2).backward()
eps = 1e-5
def _lnf(xv, wv, bv):
    mu = xv.mean(-1, keepdims=True); var = ((xv-mu)**2).mean(-1, keepdims=True)
    xh = (xv-mu) / np.sqrt(var+eps)
    return (wv * xh + bv).sum()
grad_check('layer_norm: dx',      x,  numerical_grad(lambda v: _lnf(v, w_np, b_np2), x_np.copy()))
grad_check('layer_norm: dweight', w,  numerical_grad(lambda v: _lnf(x_np, v, b_np2), w_np.copy()))
grad_check('layer_norm: dbias',   b2, numerical_grad(lambda v: _lnf(x_np, w_np, v),  b_np2.copy()))

# embedding
V_emb, D_emb = 8, 4
W_np = rng.standard_normal((V_emb, D_emb))
idx2 = np.array([0, 3, 0, 5])          # repeated 0
W2 = Tensor(W_np.copy()); embedding(W2, idx2).backward()
grad_check('embedding: dW', W2, numerical_grad(lambda w: w[idx2].sum(), W_np.copy()))

print()
print('All checks passed')


=== functional.py ===

softmax: dx                              | match: True  | maxdiff: 2.22e-11
cross_entropy: dlogits                   | match: True  | maxdiff: 2.32e-11
layer_norm: dx                           | match: True  | maxdiff: 7.85e-11
layer_norm: dweight                      | match: True  | maxdiff: 4.16e-11
layer_norm: dbias                        | match: True  | maxdiff: 4.04e-11
embedding: dW                            | match: True  | maxdiff: 5.10e-11

All checks passed
